In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [3]:
# ==========================================
# Project: Solarnex BESS Engine
# Module: Rolling Horizon Dispatch Optimizer
# ==========================================

import numpy as np
import pandas as pd
import pulp as plp
import matplotlib.pyplot as plt

# 1. Generate Synthetic Market and Load Data (Simulation Environment)
def generate_market_data(days=30):
    """
    Generates synthetic hourly electricity prices (EUR/MWh) and
    household load profiles (kW) to simulate grid edge conditions.
    """
    np.random.seed(42)
    hours = days * 24
    time_index = pd.date_range(start="2026-09-01", periods=hours, freq="h")

    # Synthetic prices with volatility and peak spikes
    base_price = 50 + 20 * np.sin(np.linspace(0, 2 * np.pi, 24))
    price_noise = np.random.normal(0, 15, hours)
    # Inject occasional price spikes (market volatility)
    spikes = np.random.choice([0, 1], size=hours, p=[0.97, 0.03]) * np.random.uniform(100, 250, hours)

    prices = np.tile(base_price, days) + price_noise + spikes
    prices = np.clip(prices, -20, 400) # Allow negative prices (common in Germany)

    # Synthetic load profile
    base_load = 1.5 + 1.0 * np.sin(np.linspace(0, 2 * np.pi, 24) - np.pi/2)
    load = np.tile(base_load, days) + np.random.normal(0, 0.3, hours)
    load = np.clip(load, 0.2, 5.0)

    df = pd.DataFrame({
        'price': prices,
        'load': load
    }, index=time_index)

    return df

# Initialize test data (starting with a 21-day window as discussed)
market_data = generate_market_data(days=21)
print(f"Dataset generated successfully. Shape: {market_data.shape}")
market_data.head()

Dataset generated successfully. Shape: (504, 2)


,price,load
2026-09-01 00:00:00,57.450712,0.580218
2026-09-01 01:00:00,53.321971,0.633592
2026-09-01 02:00:00,70.107007,0.445153
2026-09-01 03:00:00,87.462167,1.115060
2026-09-01 04:00:00,64.245404,0.987447


In [4]:
# 2. Define the BESS Optimization Engine with Rolling Horizon & Efficiency Losses
class SolarnexBESSOptimizer:
    def __init__(self, capacity_mwh=10.0, max_power_mw=2.5, rte=0.90):
        self.capacity = capacity_mwh  # Maximum energy storage capacity (MWh)
        self.max_power = max_power_mw # Max charge/discharge rate (MW)
        self.rte = rte                # Round-trip efficiency (square root applied per flow)
        self.eff_factor = np.sqrt(rte)

    def optimize_window(self, df_window):
        """
        Solves the optimal dispatch problem over a given horizon window
        using Mixed-Integer Linear Programming (MILP).
        """
        model = plp.LpProblem("Solarnex_BESS_Dispatch", plp.LpMinimize)

        T = len(df_window)
        prices = df_window['price'].values
        load = df_window['load'].values

        # Decision Variables
        # Charge power from grid (MW)
        p_charge = {t: plp.LpVariable(f"p_charge_{t}", lowBound=0, upBound=self.max_power) for t in range(T)}
        # Discharge power to grid/load (MW)
        p_discharge = {t: plp.LpVariable(f"p_discharge_{t}", lowBound=0, upBound=self.max_power) for t in range(T)}
        # State of Charge (MWh)
        soc = {t: plp.LpVariable(f"soc_{t}", lowBound=0, upBound=self.capacity) for t in range(T+1)}

        # Binary variables to prevent simultaneous charging and discharging
        is_charging = {t: plp.LpVariable(f"is_charging_{t}", cat='Binary') for t in range(T)}

        # Objective Function: Minimize total cost of grid energy purchase minus revenue
        # Cost = (Load + Charge - Discharge) * Price
        model += plp.lpSum(
            (load[t] + p_charge[t] - p_discharge[t]) * prices[t] for t in range(T)
        )

        # Initial State of Charge boundary condition
        model += soc[0] == 0.5 * self.capacity

        # Constraints loop over time steps
        for t in range(T):
            # State of Charge dynamics with Round-Trip Efficiency losses
            model += soc[t+1] == soc[t] + (p_charge[t] * self.eff_factor - p_discharge[t] / self.eff_factor) * 1.0 # 1-hour steps

            # Mutual exclusivity: cannot charge and discharge at the maximum rate simultaneously
            model += p_charge[t] <= self.max_power * is_charging[t]
            model += p_discharge[t] <= self.max_power * (1 - is_charging[t])

        # Solve model using PuLP's default solver (CBC)
        model.solve(plp.PULP_CBC_CMD(msg=0))

        # Extract results
        results = df_window.copy()
        results['p_charge'] = [p_charge[t].varValue for t in range(T)]
        results['p_discharge'] = [p_discharge[t].varValue for t in range(T)]
        results['soc'] = [soc[t+1].varValue for t in range(T)]

        return results

# Test the core optimizer on the first 21-day dataset window
optimizer = SolarnexBESSOptimizer(capacity_mwh=10.0, max_power_mw=2.5, rte=0.90)
optimized_output = optimizer.optimize_window(market_data)
print("Optimization over the initial window completed successfully.")
optimized_output[['price', 'p_charge', 'p_discharge', 'soc']].head(10)

Optimization over the initial window completed successfully.


,price,p_charge,p_discharge,soc
2026-09-01 00:00:00,57.450712,2.5,0.000000,7.371708
2026-09-01 01:00:00,53.321971,2.5,0.000000,9.743417
2026-09-01 02:00:00,70.107007,0.0,1.243416,8.432740
2026-09-01 03:00:00,87.462167,0.0,2.500000,5.797509
2026-09-01 04:00:00,64.245404,0.0,0.000000,5.797509
2026-09-01 05:00:00,66.069627,0.0,0.000000,5.797509
2026-09-01 06:00:00,93.641568,0.0,2.500000,3.162278
2026-09-01 07:00:00,80.356739,0.0,2.500000,0.527046
2026-09-01 08:00:00,59.297282,2.5,0.000000,2.898754
2026-09-01 09:00:00,70.760160,0.0,2.500000,0.263523


In [5]:
# ==========================================
# Module: Rolling Horizon Simulation & Price Spike Filter
# ==========================================

class SolarnexRollingSimulator:
    def __init__(self, full_df, optimizer, horizon_hours=24, step_hours=12):
        self.full_df = full_df
        self.optimizer = optimizer
        self.horizon = horizon_hours
        self.step = step_hours

    def apply_spike_filter(self, window_df, threshold_std=2.5):
        """
        Filters extreme price spikes or smoothes anomalies using a rolling median
        to mitigate forecasting drift effects in zero-shot / ML optimization.
        """
        rolling_median = window_df['price'].rolling(window=5, center=True, min_periods=1).median()
        rolling_std = window_df['price'].rolling(window=5, center=True, min_periods=1).std().fillna(1)

        # Identify anomalies exceeding standard deviation bounds
        upper_bound = rolling_median + threshold_std * rolling_std
        lower_bound = rolling_median - threshold_std * rolling_std

        filtered_df = window_df.copy()
        # Clip or smooth extreme spikes for robust solver execution
        filtered_df['price'] = np.clip(filtered_df['price'], lower_bound, upper_bound)
        return filtered_df

    def run_simulation(self):
        """
        Executes rolling horizon optimization across the entire dataset timeline.
        """
        total_steps = len(self.full_df)
        results_list = []

        current_soc = 0.5 * self.optimizer.capacity # Initial SoC

        for start_idx in range(0, total_steps - self.horizon, self.step):
            end_idx = start_idx + self.horizon
            window = self.full_df.iloc[start_idx:end_idx].copy()

            # Apply preprocessing filter to handle market volatility / drift
            processed_window = self.apply_spike_filter(window)

            # Run optimization for the current window
            window_result = self.optimizer.optimize_window(processed_window)

            # Store results for the forward step (e.g., first 12 hours)
            step_result = window_result.iloc[:self.step].copy()
            results_list.append(step_result)

        final_simulation_df = pd.concat(results_list)
        return final_simulation_df

# Run the rolling simulation
simulator = SolarnexRollingSimulator(market_data, optimizer, horizon_hours=48, step_hours=24)
simulation_results = simulator.run_simulation()

print(f"Rolling horizon simulation complete. Total processed rows: {len(simulation_results)}")
simulation_results[['price', 'p_charge', 'p_discharge', 'soc']].head(10)

Rolling horizon simulation complete. Total processed rows: 456


,price,p_charge,p_discharge,soc
2026-09-01 00:00:00,57.450712,2.5,0.000000,7.371708
2026-09-01 01:00:00,53.321971,2.5,0.000000,9.743417
2026-09-01 02:00:00,70.107007,0.0,1.243416,8.432740
2026-09-01 03:00:00,87.462167,0.0,2.500000,5.797509
2026-09-01 04:00:00,64.245404,0.0,0.000000,5.797509
2026-09-01 05:00:00,66.069627,0.0,0.000000,5.797509
2026-09-01 06:00:00,93.641568,0.0,2.500000,3.162278
2026-09-01 07:00:00,80.356739,0.0,2.500000,0.527046
2026-09-01 08:00:00,59.297282,2.5,0.000000,2.898754
2026-09-01 09:00:00,70.760160,0.0,2.500000,0.263523


In [6]:
import matplotlib.animation as animation

# Set dark theme styling for matplotlib
plt.style.use('dark_background')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
fig.patch.set_facecolor('#0f172a') # Dark slate background
ax1.set_facecolor('#0f172a')
ax2.set_facecolor('#0f172a')

# Subset data for smooth animation (e.g., first 168 hours / 1 week)
anim_data = simulation_results.iloc[:168]
times = anim_data.index
prices = anim_data['price'].values
p_charge = anim_data['p_charge'].values
p_discharge = anim_data['p_discharge'].values
soc = anim_data['soc'].values

# Initialize plot lines
line_price, = ax1.plot([], [], color='#38bdf8', lw=2, label='Market Price (EUR/MWh)')
ax1.set_ylabel('Price [EUR/MWh]', color='#38bdf8', fontsize=10)
ax1.tick_params(axis='y', colors='#38bdf8')
ax1.grid(True, color='#1e293b', linestyle='--', alpha=0.7)
ax1.legend(loc='upper left', frameon=True, facecolor='#1e293b', edgecolor='none')
ax1.set_title('Solarnex BESS Engine: Rolling Horizon Dispatch Simulation', color='#f8fafc', fontsize=14, pad=12)

# Power dispatch bars / lines on twin axis or separate
ax1_twin = ax1.twinx()
line_chg, = ax1_twin.plot([], [], color='#4ade80', lw=1.5, alpha=0.8, label='Charge (MW)')
line_dis, = ax1_twin.plot([], [], color='#f43f5e', lw=1.5, alpha=0.8, label='Discharge (MW)')
ax1_twin.set_ylabel('Power [MW]', color='#f8fafc', fontsize=10)
ax1_twin.tick_params(axis='y', colors='#f8fafc')
ax1_twin.set_ylim(-0.5, 3.0)
ax1_twin.legend(loc='upper right', frameon=True, facecolor='#1e293b', edgecolor='none')

# State of Charge (SoC) plot
line_soc, = ax2.plot([], [], color='#c084fc', lw=2.5, label='State of Charge (MWh)')
ax2.set_ylabel('SoC [MWh]', color='#c084fc', fontsize=10)
ax2.set_ylim(0, optimizer.capacity * 1.1)
ax2.tick_params(axis='x', colors='#94a3b8')
ax2.tick_params(axis='y', colors='#c084fc')
ax2.grid(True, color='#1e293b', linestyle='--', alpha=0.7)
ax2.legend(loc='upper left', frameon=True, facecolor='#1e293b', edgecolor='none')
ax2.set_xlabel('Timeline (Sep 2026)', color='#94a3b8', fontsize=10)

def init():
    line_price.set_data([], [])
    line_chg.set_data([], [])
    line_dis.set_data([], [])
    line_soc.set_data([], [])
    return line_price, line_chg, line_dis, line_soc

def update(frame):
    current_times = times[:frame]
    line_price.set_data(current_times, prices[:frame])
    line_chg.set_data(current_times, p_charge[:frame])
    line_dis.set_data(current_times, p_discharge[:frame])
    line_soc.set_data(current_times, soc[:frame])

    if frame > 1:
        ax1.set_xlim(times[0], times[min(frame + 10, len(times)-1)])
        ax1.set_ylim(min(prices[:frame]) - 10, max(max(prices[:frame]), 50) + 20)

    return line_price, line_chg, line_dis, line_soc

# Create animation (using Pillow writer for GIF export)
ani = animation.FuncAnimation(fig, update, frames=range(10, len(anim_data), 4),
                              init_func=init, blit=False, interval=50)

gif_path = "solarnex_bess_simulation.gif"
ani.save(gif_path, writer='pillow', fps=20, dpi=100)
plt.close(fig)

print(f"Dark theme GIF animation successfully generated and saved as: {gif_path}")

Dark theme GIF animation successfully generated and saved as: solarnex_bess_simulation.gif


In [7]:
# ==========================================
# Module: CSV Export Utility for Solarnex BESS
# ==========================================

import os

def export_simulation_to_csv(results_df, filename="solarnex_bess_simulation_output.csv"):
    """
    Exports the rolling horizon optimization results to a clean CSV file
    with formatted timestamps and operational metrics.
    """
    export_df = results_df.copy()

    # Rename columns for professional reporting standards
    export_df = export_df.rename(columns={
        'price': 'Market_Price_EUR_MWh',
        'load': 'Household_Load_MW',
        'p_charge': 'Battery_Charge_MW',
        'p_discharge': 'Battery_Discharge_MW',
        'soc': 'State_Of_Charge_MWh'
    })

    # Save to CSV
    export_df.to_csv(filename, index_label='Timestamp')
    file_size_kb = os.path.getsize(filename) / 1024

    print(f"Data successfully exported to CSV!")
    print(f"File Name: {filename}")
    print(f"File Size: {file_size_kb:.2f} KB")
    print(f"Total Rows Saved: {len(export_df)}")

# Execute CSV export
export_simulation_to_csv(simulation_results)

Data successfully exported to CSV!
File Name: solarnex_bess_simulation_output.csv
File Size: 33.10 KB
Total Rows Saved: 456
